# Cleaning Notebook

This notebook loads the raw SMS export and creates a cleaned dataset.

In [ ]:
from pathlib import Path
import pandas as pd
import re

base_dir = Path.cwd().parent
raw_path = base_dir / '1_Data_Collection' / 'raw_data.csv'
cleaned_path = base_dir / '2_Data_Cleaning' / 'cleaned_data.csv'
quality_path = base_dir / '2_Data_Cleaning' / 'data_quality_report.csv'

raw = pd.read_csv(raw_path)
raw['content'] = raw['content'].astype(str)
raw['status'] = raw['content'].str.lower().str.contains('failed|rejected|echec|echoue|annule')
raw['status'] = raw['status'].map({True: 'failed', False: 'success'})

otp_mask = raw['content'].str.contains('otp|code|login|verification', case=False, na=False)
promo_mask = raw['content'].str.contains('bonus|promo|promotion|cadeau', case=False, na=False)
cleaned = raw[~otp_mask & ~promo_mask].copy()
cleaned = cleaned.dropna(subset=['datetime', 'content'])
cleaned = cleaned.drop_duplicates(subset=['user_id', 'datetime', 'content', 'amount', 'transaction_type'])
cleaned.to_csv(cleaned_path, index=False)
quality = cleaned.isna().mean().reset_index()
quality.columns = ['column', 'missing_rate']
quality.to_csv(quality_path, index=False)
display(cleaned.head())
display(quality.head())